# PathOPEN VLM-as-Judge Runner

Runs InternVL3.5-38B and Qwen3-VL-32B-Thinking as judges over the PathOPEN core
(open-ended, MCQ, close-ended) and image-augmentation subsets, using the rubrics
defined in `data_evaluation/vlm_judge/prompts/benchmarks.py`.

Output CSVs are written in the **same column schema** as
`data_evaluation/pathologists/scoring_analysis/input/evaluatorN/*.csv` so they drop
directly into the existing `_individual.ipynb` / `_final.ipynb` scoring-analysis
notebooks as additional synthetic evaluators (`evaluator_internvl`, `evaluator_qwenvl`).

This notebook scores **every** row across all 5 evaluators' assigned parts (i.e. the
full ~231-row PathOPEN core subset and the full augmentation subset), not just one
evaluator's slice, so the judge can later be compared against whichever single
pathologist actually rated each row (see `judge_pathologist_agreement.ipynb`).

## Checkpointing (run vs. post-process are separate steps)

Each PathOPEN row requires **up to 11 separate judge calls** (2 OE correct + 2 OE
wrong + 1 MCQ correct + 4 MCQ wrong + 1 CE, each a full VLM generation). A crash or
OOM partway through a long run would lose all in-progress work if results were only
held in memory until the end.

Instead, every individual judge call is written to a JSONL checkpoint file
(`checkpoints/{model_key}_pathopen_core.jsonl` /
`checkpoints/{model_key}_pathopen_augmentation.jsonl`) **immediately** after it
completes, keyed by a stable `item_id` (`"{Image_ID}::{subtask}"`). Re-running the
scoring cell after an interruption automatically skips any `item_id` already present
in the checkpoint and only scores what's missing - no wasted GPU time re-querying
the model on items it already finished. If a single judge call raises an exception
(e.g. a generation error), that one item is logged and skipped for this pass rather
than aborting the whole run; it will be retried automatically next time the cell runs.

The final CSV-assembly step (building the evaluator-schema table) reads back
**only** from the checkpoint file, never from in-memory state - so it is always
safe to re-run on its own, even in a fresh kernel, as long as the checkpoint file
exists.


In [1]:
import os
import sys

sys.path.insert(0, os.getcwd())  # so gpu_allocation/judge_models resolve when the CWD is this dir
from gpu_allocation import cuda_visible_devices_for, describe_allocation, max_memory_for

# Which judge(s) this kernel will load. Declared HERE, before torch touches CUDA,
# because CUDA_VISIBLE_DEVICES has no effect once CUDA is initialized - if a torch CUDA
# op has already run in this kernel, restart it.
#
# Both judges run unquantized at bf16 (~64 GB Qwen / ~76 GB InternVL), so each is
# sharded over 3 of the 47.4 GiB A6000s. They are placed in different NUMA islands
# (Qwen 0-2, InternVL 4-6) so they can run concurrently without sharing a PCIe switch.
MODELS_TO_RUN = ["qwenvl", "internvl"]          # e.g. ["internvl"] or ["qwenvl", "internvl"]

os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices_for(*MODELS_TO_RUN)
print(describe_allocation())

PER_CARD_MEMORY = 42GiB
  qwenvl    -> physical GPUs [0, 1, 2]  [NUMA island A (0-3)]
  internvl  -> physical GPUs [4, 5, 6]  [NUMA island B (4-7)]
  CUDA_VISIBLE_DEVICES currently = '0,1,2,4,5,6'


In [2]:
import glob
import json as _json
import os

import pandas as pd
from PIL import Image

from checkpoint import JudgeCheckpoint
from judge_models import JudgeModel
from parallel_judges import run_judges_in_parallel
from prompts.benchmarks import (
    BENCHMARK_1,
    BENCHMARK_2,
    BENCHMARK_3,
    BENCHMARK_4,
    build_benchmark_1_prompt,
    build_benchmark_2_prompt,
    build_benchmark_3_prompt,
    build_benchmark_4_prompt,
)

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Paths

In [3]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SUBSETS_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "subsets_processing_output", "data"
)
PROCESSED_DATA_JSON = os.path.join(REPO_ROOT, "pathopen_data", "processed", "data.json")
OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
CHECKPOINT_DIR = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

REPO_ROOT, SUBSETS_DIR, PROCESSED_DATA_JSON, CHECKPOINT_DIR


('/data/mn27889/path-open-data',
 '/data/mn27889/path-open-data/data_evaluation/pathologists/subsets_processing_output/data',
 '/data/mn27889/path-open-data/pathopen_data/processed/data.json',
 '/data/mn27889/path-open-data/data_evaluation/vlm_judge/checkpoints')

### Build Image_ID -> file path resolver

`processed/data.json` maps every `img_pathopen_<case>_<imgnum>` ID to its directory,
original filename, and list of augmented filenames. This is more reliable than
resolving through the Google Drive links used to hand images to human evaluators,
since all files already exist locally.

Two cases (9 and 79) have an empty `Original` field in `processed/data.json` — a
pre-existing gap in the processed dataset (their raw images were never carried
through `raw_pathopen_data_processing.ipynb`), not something introduced here. Rows
referencing these are skipped and reported at the end of each scoring pass rather
than silently dropped.

In [4]:
with open(PROCESSED_DATA_JSON) as f:
    _processed_cases = _json.load(f)

_processed_root = os.path.dirname(PROCESSED_DATA_JSON)  # .../pathopen_data/processed

# image_id -> {dir, original filename, [augmented filenames]}
IMAGE_INDEX = {}
CASES_MISSING_IMAGES = []
for _case in _processed_cases:
    for _img in _case["Images"]:
        original = _img.get("Original", "")
        if not original:
            CASES_MISSING_IMAGES.append(_case["Case ID"])
            continue
        image_id = original.rsplit("_orig.", 1)[0]
        # _img["Directory"] is like '../pathopen_data/processed/images/img_pathopen_153_03';
        # re-anchor everything from 'images/...' onward under _processed_root.
        rel_dir = _img["Directory"].split("processed/")[-1]
        abs_dir = os.path.normpath(os.path.join(_processed_root, rel_dir))
        IMAGE_INDEX[image_id] = {
            "dir": abs_dir,
            "original": original,
            "augmented": _img.get("Augmented", []),
        }

print(f"{len(IMAGE_INDEX)} images indexed; cases with no resolvable image: {CASES_MISSING_IMAGES}")


229 images indexed; cases with no resolvable image: [9, 79]


In [5]:
def load_original_image(image_id: str) -> Image.Image:
    entry = IMAGE_INDEX[image_id]
    path = os.path.join(entry["dir"], entry["original"])
    return Image.open(path).convert("RGB")


def load_augmented_image(augmented_filename: str) -> Image.Image:
    """augmented_filename looks like 'img_pathopen_148_01_aug_0.png'; derive the base
    image_id by stripping the '_aug_N.ext' suffix to find its directory."""
    base = augmented_filename
    if "_aug_" in base:
        image_id = base.split("_aug_")[0]
    else:
        image_id = base
    entry = IMAGE_INDEX[image_id]
    path = os.path.join(entry["dir"], augmented_filename)
    return Image.open(path).convert("RGB")


## Load judge models

Both judges run **unquantized at bf16**. 4-bit NF4 was found to corrupt InternVL's
vision path badly enough that it described a completely different image than the one
supplied (histology read back as "Barbie logos", "a tweet screenshot"), and running both
judges at the same precision keeps a judge-vs-judge comparison free of a quantization
confound. See the module docstring in `judge_models.py`.

Per-judge behaviour (system role, reasoning prompt, image-passing convention) is declared
in `JUDGE_SPECS` in `judge_models.py` — notably InternVL needs an R1-style system prompt
to enable reasoning at all, which materially changes its scores on Benchmark 2.

GPU placement comes from `gpu_allocation.py`: each judge is sharded across 3 cards inside
one NUMA island (Qwen 0-2, InternVL 4-6), so both can be resident and run concurrently.
`MODELS_TO_RUN` is set in the **first cell** of this notebook, since GPU visibility must
be decided before CUDA initializes.

In [6]:
# MODELS_TO_RUN is set in the first cell (it has to be, to pick GPUs before CUDA init).
# max_memory_for(...) returns LOGICAL device ids - CUDA_VISIBLE_DEVICES renumbers cards,
# so with "4,5,6" visible torch sees 0,1,2.
judges = {
    key: JudgeModel(key, max_memory=max_memory_for(key, MODELS_TO_RUN))
    for key in MODELS_TO_RUN
}

# Verify each judge loaded as intended BEFORE committing to a multi-hour run.
# device_map="auto" silently offloads to CPU/disk when a model does not fit, which turns
# a ~10-hour run into a multi-day one - catch it here, not six hours in.
import collections

import torch

for key, judge in judges.items():
    devs = collections.Counter(str(p.device) for p in judge.model.parameters())
    dtypes = collections.Counter(str(p.dtype) for p in judge.model.parameters())
    offloaded = [d for d in devs if d in ("cpu", "meta", "disk")]
    total = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1024**3
    print(f"{key}: dtypes={dict(dtypes)}")
    print(f"    devices={dict(devs)}")
    print(f"    system_role={judge.use_system_role}  images_in_template={judge.images_in_template}")
    print(f"    thinking prompt: {'<think>' in judge.system_prompt}")
    print(f"    OFFLOAD -> {offloaded if offloaded else 'none (good)'}")
    if offloaded:
        raise RuntimeError(f"{key} offloaded to {offloaded} - lower max_memory or add a card")

print(f"\ntotal VRAM allocated across visible cards: {total:.1f} GiB")
judges

Loading weights: 100%|██████████| 1392/1392 [00:19<00:00, 70.01it/s] 
[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


qwenvl: dtypes={'torch.bfloat16': 1058}
    devices={'cuda:0': 561, 'cuda:1': 264, 'cuda:2': 233}
    system_role=False  images_in_template=False
    thinking prompt: False
    OFFLOAD -> none (good)
internvl: dtypes={'torch.bfloat16': 1392}
    devices={'cuda:3': 819, 'cuda:4': 286, 'cuda:5': 287}
    system_role=True  images_in_template=True
    thinking prompt: True
    OFFLOAD -> none (good)

total VRAM allocated across visible cards: 133.7 GiB


{'qwenvl': <judge_models.JudgeModel at 0x7f8a27789550>,
 'internvl': <judge_models.JudgeModel at 0x7f8972177110>}

## Score the PathOPEN core subset (OE, MCQ, CE)

Iterates all `pathopen_vqa_part{1-5}.csv` files (the full pathologist-facing subset,
not a single evaluator's slice). For each row, up to 11 individual judge calls are
made (one per question/answer/option needing a score):
- OE correct answers (x2 per case) -> Benchmark 1
- OE wrong answers (x2 per case, where present) -> Benchmark 2
- MCQ correct option (as open-ended) -> Benchmark 1
- MCQ wrong options (x4) -> Benchmark 2
- CE question/answer -> Benchmark 3

Each call is checkpointed individually (`item_id = "{Image_ID}::{subtask}"`), so a
crash mid-row only loses the one in-flight call, not the rest of that row's
progress.

In [7]:
core_parts = sorted(glob.glob(os.path.join(SUBSETS_DIR, "pathopen_vqa_part*.csv")))
core_df = pd.concat([pd.read_csv(p) for p in core_parts], ignore_index=True)
resolvable_core_df = core_df[core_df["Image_ID"].isin(IMAGE_INDEX)]
skipped_core_rows = core_df[~core_df["Image_ID"].isin(IMAGE_INDEX)]
if len(skipped_core_rows):
    print(f"Skipping {len(skipped_core_rows)} row(s) with no resolvable image:")
    print(skipped_core_rows[["Case_ID", "Image_ID"]])

len(core_df), len(resolvable_core_df)


Skipping 2 row(s) with no resolvable image:
     Case_ID            Image_ID
93        79  img_pathopen_79_01
173        9   img_pathopen_9_01


(231, 229)

### Sub-tasks per row

`iter_core_subtasks(row)` yields `(subtask_name, prompt, criteria, criterion_key_map)`
for every judge call a row requires. `criterion_key_map` maps the rubric's criterion
name (as returned by `judge.score`) to the flat field name it should end up under in
the final CSV row.

In [8]:
def iter_core_subtasks(row: pd.Series):
    for i in (1, 2):
        q_col, correct_col, wrong_col = f"OE_Question_{i}", f"OE_Correct_Answer_{i}", f"OE_Wrong_Answer_{i}"
        yield (
            f"OE_Correct_{i}",
            build_benchmark_1_prompt(row[q_col], row[correct_col]),
            list(BENCHMARK_1["criteria"].keys()),
            {
                "Knowledge Interpretation/Deduction": f"Evaluation OE_Correct_Answer_{i}\n(Benchmark 1)",
                "Visual Grounding": f"OE_Correct_Answer_{i}_VisGround",
            },
        )
        wrong_answer = row.get(wrong_col)
        if pd.notna(wrong_answer):
            yield (
                f"OE_Wrong_{i}",
                build_benchmark_2_prompt(row[q_col], row[correct_col], wrong_answer),
                list(BENCHMARK_2["criteria"].keys()),
                {
                    "Error Proximity and Deductive Plausibility": f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
                    "Visual Grounding Error": f"OE_Wrong_Answer_{i}_VisGroundErr",
                },
            )

    yield (
        "MCQ_Correct",
        build_benchmark_1_prompt(row["MCQ_OE_Question"], row["MCQ_OE_Correct_Answer"]),
        list(BENCHMARK_1["criteria"].keys()),
        {
            "Knowledge Interpretation/Deduction": "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)",
            "Visual Grounding": "MCQ_OE_Correct_Answer_VisGround",
        },
    )

    for i in (1, 2, 3, 4):
        wrong_col = f"MCQ_OE_Wrong_Answer_{i}"
        yield (
            f"MCQ_Wrong_{i}",
            build_benchmark_2_prompt(row["MCQ_OE_Question"], row["MCQ_OE_Correct_Answer"], row[wrong_col]),
            list(BENCHMARK_2["criteria"].keys()),
            {
                "Error Proximity and Deductive Plausibility": f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
                "Visual Grounding Error": f"MCQ_OE_Wrong_Answer_{i}_VisGroundErr",
            },
        )

    yield (
        "CE_Correct",
        build_benchmark_3_prompt(row["CE_Question"], row["CE_Correct_Answer"]),
        list(BENCHMARK_3["criteria"].keys()),
        {"Visual Grounding/Reasoning": "Evaluation CE_Correct_Answer\n(Benchmark 3)"},
    )


### Run: score every (row, subtask) pair, checkpointing each call

Skips any `item_id` already present in the checkpoint file from a previous run.

In [9]:
def run_core_scoring(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathopen_core.jsonl"))
    n_scored = n_skipped = n_failed = 0

    from tqdm.auto import tqdm
    for _, row in tqdm(resolvable_core_df.iterrows(), total=len(resolvable_core_df), desc=f"{model_key} core"):
        image = None  # lazily loaded only if at least one subtask for this row is missing
        for subtask_name, prompt, criteria, key_map in iter_core_subtasks(row):
            item_id = f"{row['Image_ID']}::{subtask_name}"
            print(f"[{n_scored}/{len(resolvable_core_df)}][checkpoint] {model_key} core: scoring item_id={item_id!r}")
            if checkpoint.is_done(item_id):
                n_skipped += 1
                continue
            try:
                if image is None:
                    image = load_original_image(row["Image_ID"])
                scores, raw = judge.score(image, prompt, criteria)
            except Exception as e:
                n_failed += 1
                print(f"[{n_scored}/{len(resolvable_core_df)}][checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
                continue
            checkpoint.append({
                "item_id": item_id,
                "Case_ID": row["Case_ID"],
                "Image_ID": row["Image_ID"],
                "Image_URL": row["Image_URL"],
                "subtask": subtask_name,
                "scores": scores,
                "key_map": key_map,
                "raw_response": raw,
            })
            n_scored += 1
            print(f"[{n_scored}/{len(resolvable_core_df)}][checkpoint] {model_key} core: scored item_id={item_id!r} - {scores}")

    print(f"{model_key} core: scored {n_scored} new, {n_skipped} already done, {n_failed} failed this run")
    return checkpoint


# Judges run CONCURRENTLY, not one after the other. Each has its own GPUs and its own
# checkpoint file, and model.generate() releases the GIL during CUDA work, so two judges
# genuinely overlap: ~24h instead of ~46h for Qwen + InternVL. See parallel_judges.py.
core_checkpoints = run_judges_in_parallel(run_core_scoring, judges, MODELS_TO_RUN)


[parallel] starting 2 judge(s) concurrently: ['qwenvl', 'internvl']
[parallel]   qwenvl -> /data/mn27889/path-open-data/data_evaluation/vlm_judge/logs/qwenvl_run_core_scoring_20260808_112828.log
[parallel]   internvl -> /data/mn27889/path-open-data/data_evaluation/vlm_judge/logs/internvl_run_core_scoring_20260808_112828.log
[parallel]   follow with:  tail -f <path>


qwenvl core:   0%|          | 0/229 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Post-process: assemble the checkpoint into the evaluator-schema CSV

Pure re-read of the checkpoint **file** (via a fresh `JudgeCheckpoint(path)`, not
the in-memory `core_checkpoints` object from the scoring cell above) - safe to
re-run any time, independent of the scoring cell above, even after restarting the
kernel. Only needs `resolvable_core_df` in scope for the question/answer text
columns (re-run the "Score the PathOPEN core subset" data cell if starting fresh -
it's instant, no GPU involved).

In [ ]:
CORE_TEXT_COLUMNS = [
    "OE_Question_1", "OE_Correct_Answer_1", "OE_Wrong_Answer_1",
    "OE_Question_2", "OE_Correct_Answer_2", "OE_Wrong_Answer_2",
    "MCQ_OE_Question", "MCQ_OE_Correct_Answer",
    "MCQ_OE_Wrong_Answer_1", "MCQ_OE_Wrong_Answer_2", "MCQ_OE_Wrong_Answer_3", "MCQ_OE_Wrong_Answer_4",
    "CE_Question", "CE_Correct_Answer",
]

# Preserves the original column ORDER of pathopen_eval_data.csv (text column
# immediately followed by its score column(s)) so the output is directly
# comparable to the human evaluator CSVs, not just schema-equivalent.
CORE_COLUMN_ORDER = [
    "CASE_ID", "Image_ID", "Image_URL",
    "OE_Question_1", "OE_Correct_Answer_1",
    "Evaluation OE_Correct_Answer_1\n(Benchmark 1)", "OE_Correct_Answer_1_VisGround",
    "OE_Wrong_Answer_1",
    "Evaluation OE_Wrong_Answer_1\n(Benchmark 2)", "OE_Wrong_Answer_1_VisGroundErr",
    "OE_Question_2", "OE_Correct_Answer_2",
    "Evaluation OE_Correct_Answer_2\n(Benchmark 1)", "OE_Correct_Answer_2_VisGround",
    "OE_Wrong_Answer_2",
    "Evaluation OE_Wrong_Answer_2\n(Benchmark 2)", "OE_Wrong_Answer_2_VisGroundErr",
    "MCQ_OE_Question", "MCQ_OE_Correct_Answer",
    "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)", "MCQ_OE_Correct_Answer_VisGround",
    "MCQ_OE_Wrong_Answer_1",
    "Evaluation MCQ_OE_Wrong_Answer_1\n(Benchmark 2)", "MCQ_OE_Wrong_Answer_1_VisGroundErr",
    "MCQ_OE_Wrong_Answer_2",
    "Evaluation MCQ_OE_Wrong_Answer_2\n(Benchmark 2)", "MCQ_OE_Wrong_Answer_2_VisGroundErr",
    "MCQ_OE_Wrong_Answer_3",
    "Evaluation MCQ_OE_Wrong_Answer_3\n(Benchmark 2)", "MCQ_OE_Wrong_Answer_3_VisGroundErr",
    "MCQ_OE_Wrong_Answer_4",
    "Evaluation MCQ_OE_Wrong_Answer_4\n(Benchmark 2)", "MCQ_OE_Wrong_Answer_4_VisGroundErr",
    "CE_Question", "CE_Correct_Answer",
    "Evaluation CE_Correct_Answer\n(Benchmark 3)",
]


def assemble_core_csv(model_key: str, source_df: pd.DataFrame) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_pathopen_core.jsonl directly from disk - does
    NOT depend on the `core_checkpoints` dict from the scoring cell, so this is
    safe to run standalone in a fresh kernel."""
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathopen_core.jsonl"))
    text_by_image_id = source_df.set_index("Image_ID")[CORE_TEXT_COLUMNS].to_dict(orient="index")

    rows_by_image_id = {}
    for record in checkpoint.load_all():
        image_id = record["Image_ID"]
        row = rows_by_image_id.setdefault(image_id, {
            "CASE_ID": record["Case_ID"],
            "Image_ID": image_id,
            "Image_URL": record["Image_URL"],
            **text_by_image_id.get(image_id, {}),
        })
        for criterion, field_name in record["key_map"].items():
            row[field_name] = record["scores"].get(criterion)

    out_df = pd.DataFrame.from_records(list(rows_by_image_id.values()))
    return out_df.reindex(columns=CORE_COLUMN_ORDER)


for model_key in MODELS_TO_RUN:
    out_df = assemble_core_csv(model_key, resolvable_core_df)
    out_path = os.path.join(OUTPUT_DIR, f"evaluator_{model_key}", "pathopen_eval_data.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(model_key, "->", out_path, out_df.shape)


## Score the image-augmentation subset

Reuses each case's original open-ended Q&A pair against its augmented image(s),
scored with Benchmark 4 (Clinical Relevance, Visual Grounding). Output mirrors
`pathopen_image_augmentation_eval_data.csv`'s schema. Same checkpoint-then-assemble
pattern as the core subset above (2 subtasks per row: `OE_1`, `OE_2`).

In [ ]:
aug_parts = sorted(glob.glob(os.path.join(SUBSETS_DIR, "pathopen_vqa_augmented_part*.csv")))
aug_df = pd.concat([pd.read_csv(p) for p in aug_parts], ignore_index=True)
len(aug_df), aug_parts


In [ ]:
def iter_augmentation_subtasks(row: pd.Series):
    for i in (1, 2):
        q_col, a_col = f"Open Ended - Question {i}", f"Open Ended - Answer {i}"
        suffix = "" if i == 1 else ".1"
        yield (
            f"OE_{i}",
            build_benchmark_4_prompt(row[q_col], row[a_col]),
            list(BENCHMARK_4["criteria"].keys()),
            {
                "Clinical Relevance": f"Evaluation Image_Augmentation\n(Benchmark 4){suffix}",
                "Visual Grounding": f"OE_Image_Augmentation_{i}_VisGround",
            },
        )


def run_augmentation_scoring(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathopen_augmentation.jsonl"))
    n_scored = n_skipped = n_failed = 0

    from tqdm.auto import tqdm
    for _, row in tqdm(aug_df.iterrows(), total=len(aug_df), desc=f"{model_key} augmentation"):
        image = None
        for subtask_name, prompt, criteria, key_map in iter_augmentation_subtasks(row):
            item_id = f"{row['Augmented_Image_ID']}::{subtask_name}"
            print(f"[{n_scored}/{len(aug_df)}][checkpoint] {model_key} augmentation: scoring item_id={item_id!r}")
            if checkpoint.is_done(item_id):
                n_skipped += 1
                continue
            try:
                if image is None:
                    image = load_augmented_image(row["Augmented_Image_ID"])
                scores, raw = judge.score(image, prompt, criteria)
            except Exception as e:
                n_failed += 1
                print(f"[{n_scored}/{len(aug_df)}][checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
                continue
            checkpoint.append({
                "item_id": item_id,
                "Case_ID": row["Case_ID"],
                "Image_ID": row["Augmented_Image_ID"],
                "Image_URL": row["Augmented_Image_Link"],
                "subtask": subtask_name,
                "scores": scores,
                "key_map": key_map,
                "raw_response": raw,
            })
            n_scored += 1
            print(f"[{n_scored}/{len(aug_df)}][checkpoint] {model_key} augmentation: scored item_id={item_id!r} - {scores}")
            
    print(f"{model_key} augmentation: scored {n_scored} new, {n_skipped} already done, {n_failed} failed this run")
    return checkpoint


aug_checkpoints = run_judges_in_parallel(run_augmentation_scoring, judges, MODELS_TO_RUN)


In [ ]:
AUGMENTATION_TEXT_COLUMNS = [
    "Open Ended - Question 1", "Open Ended - Answer 1",
    "Open Ended - Question 2", "Open Ended - Answer 2",
]

AUGMENTATION_COLUMN_ORDER = [
    "CASE_ID", "Image_ID", "Image_URL",
    "OE_Question_1", "OE_Correct_Answer_1",
    "Evaluation Image_Augmentation\n(Benchmark 4)", "OE_Image_Augmentation_1_VisGround",
    "OE_Question_2", "OE_Correct_Answer_2",
    "Evaluation Image_Augmentation\n(Benchmark 4).1", "OE_Image_Augmentation_2_VisGround",
]


def assemble_augmentation_csv(model_key: str, source_df: pd.DataFrame) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_pathopen_augmentation.jsonl directly from disk -
    does NOT depend on the `aug_checkpoints` dict from the scoring cell, so this is
    safe to run standalone in a fresh kernel."""
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathopen_augmentation.jsonl"))
    text_by_image_id = source_df.set_index("Augmented_Image_ID")[AUGMENTATION_TEXT_COLUMNS].to_dict(orient="index")

    rows_by_image_id = {}
    for record in checkpoint.load_all():
        image_id = record["Image_ID"]
        text = text_by_image_id.get(image_id, {})
        row = rows_by_image_id.setdefault(image_id, {
            "CASE_ID": record["Case_ID"],
            "Image_ID": image_id,
            "Image_URL": record["Image_URL"],
            "OE_Question_1": text.get("Open Ended - Question 1"),
            "OE_Correct_Answer_1": text.get("Open Ended - Answer 1"),
            "OE_Question_2": text.get("Open Ended - Question 2"),
            "OE_Correct_Answer_2": text.get("Open Ended - Answer 2"),
        })
        for criterion, field_name in record["key_map"].items():
            row[field_name] = record["scores"].get(criterion)

    out_df = pd.DataFrame.from_records(list(rows_by_image_id.values()))
    return out_df.reindex(columns=AUGMENTATION_COLUMN_ORDER)


for model_key in MODELS_TO_RUN:
    out_df = assemble_augmentation_csv(model_key, aug_df)
    out_path = os.path.join(OUTPUT_DIR, f"evaluator_{model_key}", "pathopen_image_augmentation_eval_data.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(model_key, "->", out_path, out_df.shape)


## Next steps

The resulting `judge_output/evaluator_internvl/` and `judge_output/evaluator_qwenvl/`
directories can be copied into
`data_evaluation/pathologists/scoring_analysis/input/` alongside `evaluator{1-5}/` and
run through the existing `pathopen_scoring_analysis_individual.ipynb` /
`pathopen_image_augmentation_scoring_analysis_individual.ipynb` notebooks unchanged
to get per-judge score-level distribution plots.

The raw checkpoint files in `checkpoints/*.jsonl` retain every model response
(including the full raw text, for auditing/debugging) and are never overwritten -
safe to keep around after the CSVs are built, and re-running the assembly cells
above is always safe even without re-running any scoring.

For judge-vs-pathologist agreement (kappa, Mann-Whitney U), see
`judge_pathologist_agreement.ipynb`.